# BiLSTM Arabic Diacritization Pipeline (PyTorch)

This notebook implements a character-level BiLSTM pipeline for Arabic text diacritization using PyTorch.
Edit the `DATA_PATH` variables in the first code cell to point to your dataset.

* The only actual input feature to the model is:

        A 25-dimensional embedding vector representing each character.

* For each position in the sequence:

    1) Take the character ID (like "ل", "س", "و", <SPACE>, <NONAR>, etc.)

    2) Convert it to a 25-dimensional vector using the embedding layer.

    3) Feed these 25-dim vectors into the BiLSTM from left-to-right and right-to-left.

* So the network learns:

    1) context from characters before the position (forward LSTM)

    2) context from characters after the position (backward LSTM)

    Then it merges the two directions → a contextual representation of the character.

    Finally, the model outputs a diacritic class for that timestep.

* Arabic diacritization is context-dependent, e.g.

    1) “عَلِمَ” vs “عُلِمَ” (vowel depends on verb form)

    2) “في البيتِ” vs “في البيتُ” (case endings depend on grammar)

    3) with Shadda or without depending on morphological assimilation

* The BiLSTM captures these dependencies because:

    1) embeddings encode the character identity

    2) LSTM hidden states encode the sequence context


In [ ]:

import os
import glob
import unicodedata
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import tarfile


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

COMPRESSED_FILE = "/kaggle/input/tashkeela/Tashkeela-arabic-diacritized-text-utf8-0.3.tar.bz2"
EXTRACT_DIR = "/kaggle/working/tashkeela_extracted/"
PRETRAINED_PATH = "/kaggle/input/existing/pytorch/default/1/bilstm_Train_with_Val.pt"
OUTPUT_MODEL_PATH = "/kaggle/working/bilstm_finetuned.pt"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Hyperparameters 
# ------------------------------------------------------------
MAXLEN = 500
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 64
EPOCHS = 1
LR = 5e-4

MAX_TRAIN_LINES = 500000
MAX_VAL_LINES = 1000

PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
SPACE_TOKEN = "<SPACE>"
PLACEHOLDER = "<NONAR>"

ckpt = torch.load(PRETRAINED_PATH, map_location=device)

char2idx = ckpt["char2idx"]
diac2idx = ckpt["diac2idx"]

idx2char = {i:c for c,i in char2idx.items()}
idx2diac = {i:d for d,i in diac2idx.items()}

VOCAB_SIZE = len(char2idx)
NUM_LABELS = len(diac2idx)
EMBED_DIM = ckpt["model_state_dict"]["embedding.weight"].shape[1]
pad_label_idx = diac2idx["<PAD_LABEL>"]

print("Loaded vocab:", VOCAB_SIZE)
print("Embedding dim:", EMBED_DIM)
print("Labels:", NUM_LABELS)

if not os.path.exists(EXTRACT_DIR):
    with tarfile.open(COMPRESSED_FILE, "r:bz2") as tar:
        tar.extractall(EXTRACT_DIR)

DATA_PATH = os.path.join(EXTRACT_DIR, "**", "*.txt")

# ------------------------------------------------------------
# Arabic preprocessing
# ------------------------------------------------------------
def is_combining(ch):
    return unicodedata.category(ch) == "Mn"

def is_arabic_letter(ch):
    try:
        return "ARABIC" in unicodedata.name(ch)
    except:
        return False

def normalize_arabic(text):
    return (
        text.replace("أ","ا")
            .replace("إ","ا")
            .replace("آ","ا")
            .replace("ى","ي")
            .replace("ؤ","و")
            .replace("ئ","ي")
    )

def split_char_diacritic_pairs(sentence):
    pairs, base, diacs = [], None, ""
    for ch in sentence:
        if is_combining(ch):
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base, diacs = ch, ""
    if base is not None:
        pairs.append((base, diacs))
    return pairs

def preprocess_sentence(sentence):
    tokens, labels = [], []
    for base, d in split_char_diacritic_pairs(sentence):
        if base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append("")
        elif is_arabic_letter(base) or base == "ـ":
            tokens.append(normalize_arabic(base))
            labels.append(d)
        else:
            tokens.append(PLACEHOLDER)
            labels.append("")
    return tokens, labels

# ------------------------------------------------------------
# Load Tashkeela
# ------------------------------------------------------------
def load_tashkeela(path):
    lines = []
    for f in glob.glob(path, recursive=True):
        if not os.path.isfile(f):
            continue
        with open(f, encoding="utf-8") as fh:
            for l in fh:
                l = l.strip()
                if len(l) > 5:
                    lines.append(l)
    return lines

all_lines = load_tashkeela(DATA_PATH)
split = int(0.9 * len(all_lines))

train_lines = all_lines[:split][:MAX_TRAIN_LINES]
val_lines = all_lines[split:][:MAX_VAL_LINES]

print("Train lines:", len(train_lines))
print("Val lines:", len(val_lines))

# ------------------------------------------------------------
# Preprocess
# ------------------------------------------------------------
train_tokens, train_labels = zip(*(preprocess_sentence(l) for l in train_lines))
val_tokens, val_labels = zip(*(preprocess_sentence(l) for l in val_lines))

# ------------------------------------------------------------
# Encode
# ------------------------------------------------------------
def encode(tokens, labels):
    X, Y = [], []
    for t,l in zip(tokens,labels):
        t = [SOS_TOKEN] + list(t) + [EOS_TOKEN]
        l = [""] + list(l) + [""]

        x = [char2idx.get(c, char2idx[UNK_TOKEN]) for c in t][:MAXLEN]
        y = [diac2idx.get(d, diac2idx["<NONE>"]) for d in l][:MAXLEN]

        X.append(x + [char2idx[PAD_TOKEN]]*(MAXLEN-len(x)))
        Y.append(y + [pad_label_idx]*(MAXLEN-len(y)))

    return np.array(X), np.array(Y)

X_train, y_train = encode(train_tokens, train_labels)
X_val, y_val = encode(val_tokens, val_labels)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class DiacDataset(Dataset):
    def __init__(self,X,Y):
        self.X,self.Y = X,Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self,i):
        return torch.tensor(self.X[i]), torch.tensor(self.Y[i])

train_loader = DataLoader(DiacDataset(X_train,y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(DiacDataset(X_val,y_val), batch_size=BATCH_SIZE)

# ------------------------------------------------------------
# Bilstm Model
# ------------------------------------------------------------
class BiLSTM_Diac(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=char2idx[PAD_TOKEN])

        self.bilstm1 = nn.LSTM(EMBED_DIM, LSTM_UNITS, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(DROPOUT)

        self.bilstm2 = nn.LSTM(2*LSTM_UNITS, LSTM_UNITS, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(DROPOUT)

        self.ff1 = nn.Linear(2*LSTM_UNITS, FF_UNITS)
        self.ff2 = nn.Linear(FF_UNITS, FF_UNITS)
        self.out = nn.Linear(FF_UNITS, NUM_LABELS)

        self.relu = nn.ReLU()

    def forward(self, x):
        emb = self.embedding(x)
        o1,_ = self.bilstm1(emb)
        o1 = self.dropout1(o1)
        o2,_ = self.bilstm2(o1)
        o2 = self.dropout2(o2)
        ff = self.relu(self.ff1(o2))
        ff = self.relu(self.ff2(ff))
        return self.out(ff)

model = BiLSTM_Diac().to(device)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
print("Pretrained weights loaded")

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
def compute_accuracy(logits, targets, pad_idx):
    preds = logits.argmax(dim=-1)
    mask = targets != pad_idx
    correct = (preds[mask] == targets[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

def compute_der(logits, targets, pad_idx):
    preds = logits.argmax(dim=-1)
    mask = targets != pad_idx
    errors = (preds[mask] != targets[mask]).sum().item()
    total = mask.sum().item()
    return errors / total if total > 0 else 0.0

# ------------------------------------------------------------
# Optimizer & Loss
# ------------------------------------------------------------
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=pad_label_idx)

# ------------------------------------------------------------
# Training Loop
# ------------------------------------------------------------
best_der = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for X,Y in train_loader:
        X,Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits.view(-1, NUM_LABELS), Y.view(-1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        torch.save(
            {"epoch": epoch, "model_state_dict": model.state_dict()},
            os.path.join(CHECKPOINT_DIR, "last_batch.pt")
        )

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    val_der = 0.0
    batches = 0

    with torch.no_grad():
        for X,Y in val_loader:
            X,Y = X.to(device), Y.to(device)
            logits = model(X)

            loss = criterion(logits.view(-1, NUM_LABELS), Y.view(-1))
            val_loss += loss.item()

            val_acc += compute_accuracy(logits, Y, pad_label_idx)
            val_der += compute_der(logits, Y, pad_label_idx)
            batches += 1

    val_loss /= batches
    val_acc /= batches
    val_der /= batches

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.4f} | "
        f"Val DER {val_der:.4f}"
    )

    if val_der < best_der:
        best_der = val_der
        torch.save(
            {"epoch": epoch, "model_state_dict": model.state_dict(), "val_der": val_der},
            OUTPUT_MODEL_PATH
        )
        print("⭐ Best model saved (lowest DER)")

print("Fine-tuning completed successfully")


Using device: cuda
Loaded vocab: 45
Embedding dim: 25
Labels: 16
Train lines: 500000
Val lines: 1000
Pretrained weights loaded
Epoch 01 | Train Loss 0.1098 | Val Loss 0.0494 | Val Acc 0.9834 | Val DER 0.0166
⭐ Best model saved (lowest DER)
Fine-tuning completed successfully
